# ARS Notebook 04 — ARC-Challenge Multiple-Choice Transfer

Evaluate the four paired arms on **500 ARC-Challenge test questions**. The notebook runs 0-shot first, writes its standalone ZIP, then runs 2-shot and writes a separate ZIP. This benchmark tests transfer to science reasoning in a multiple-choice format.


## Required inputs

Attach one complete Stage B ars_training_bundle.zip or one extracted Stage B input tree with its root manifest.json. It must include the fixed adapter, route bank, router, norm calibrations, and validation-selected static-bank action. No Stage A bundle, old checkpoint, or rollout files are needed.

For a resumed Kaggle session, you may also attach either or both result ZIPs produced by this notebook. The notebook verifies provenance, reuses completed shot conditions, and emits separate ZIPs for each shot count.


In [ ]:
import hashlib, json, os, shutil, subprocess, sys
from pathlib import Path

os.environ["TOKENIZERS_PARALLELISM"] = "false"
REPO_URL = "https://github.com/Abdullah182155/adaptive-residual-steering.git"
REPO_DIR = Path("/kaggle/working/adaptive-residual-steering")
WORK_ROOT = Path("/kaggle/working")
INPUT_ROOT = Path("/kaggle/input")
if (REPO_DIR / ".git").exists():
    subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "origin", "development"], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "checkout", "development"], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only", "origin", "development"], check=True)
else:
    subprocess.run(["git", "clone", "--branch", "development", REPO_URL, str(REPO_DIR)], check=True)
os.chdir(REPO_DIR)
sys.path.insert(0, str(REPO_DIR))
from scripts.research_notebook_utils import (
    extract_zip_once, locate_named_file, merge_result_bundle, pack_files,
    resolve_artifacts, run_live, sha256_file, atomic_write_json,
)

# Keep the training and evaluation revisions explicit. A stale script snapshot
# inside the training bundle must not replace the current benchmark runner.
EVALUATION_CODE_COMMIT = subprocess.run(
    ["git", "-C", str(REPO_DIR), "rev-parse", "HEAD"],
    check=True, capture_output=True, text=True,
).stdout.strip()
training_zips = sorted(INPUT_ROOT.rglob("ars_training_bundle.zip"))
if len(training_zips) > 1:
    raise ValueError(f"Attach one training bundle only; found: {[str(p) for p in training_zips]}")
TRAINING_ZIP = training_zips[0] if training_zips else None
TRAINING_INPUT = None
TRAINING_MANIFEST = {}
if TRAINING_ZIP:
    ARTIFACT_ROOT = extract_zip_once(TRAINING_ZIP, WORK_ROOT / "ars_training_artifacts")
    TRAINING_INPUT = TRAINING_ZIP
    manifest_path = ARTIFACT_ROOT / "manifest.json"
    if manifest_path.is_file():
        TRAINING_MANIFEST = json.loads(manifest_path.read_text(encoding="utf-8"))
else:
    manifests = []
    for candidate in INPUT_ROOT.rglob("manifest.json"):
        try:
            data = json.loads(candidate.read_text(encoding="utf-8"))
        except (OSError, ValueError, TypeError):
            continue
        if data.get("format") == "ars_training_bundle_v1":
            manifests.append((candidate, data))
    if len(manifests) > 1:
        raise ValueError(f"Attach one extracted ARS training bundle only; found: {[str(p) for p, _ in manifests]}")
    if manifests:
        manifest_path, TRAINING_MANIFEST = manifests[0]
        ARTIFACT_ROOT = manifest_path.parent
        TRAINING_INPUT = ARTIFACT_ROOT
    else:
        ARTIFACT_ROOT = INPUT_ROOT
        manifest_path = None

REQUIRED_ARTIFACTS = [
    "fixed_route_adapter.pt", "session3_steering_bank.pt", "session3_router.pt",
    "fixed_route_p99_norms.json", "candidate_norms_p99.json",
    "session3_static_bank_action.json",
]
try:
    ARTIFACTS = resolve_artifacts(ARTIFACT_ROOT, REQUIRED_ARTIFACTS)
    INPUTS_READY = bool(TRAINING_INPUT)
except Exception as exc:
    ARTIFACTS, INPUTS_READY = {}, False
    print("Training bundle is missing or invalid:", exc, flush=True)
if not TRAINING_INPUT:
    INPUTS_READY = False
    print("Attach ars_training_bundle.zip or one extracted input containing its root manifest.json.", flush=True)
elif (
    not TRAINING_MANIFEST.get("training_ready", False)
    or TRAINING_MANIFEST.get("architecture_id") != "ars.fixed_route_low_rank_residual"
    or TRAINING_MANIFEST.get("architecture_version") != "1.0.0"
):
    INPUTS_READY = False
    print("Training bundle is not ready or has an unsupported fixed-adapter architecture.", TRAINING_MANIFEST.get("required_phase_status"), flush=True)
TRAINING_CODE_COMMIT = TRAINING_MANIFEST.get("code_commit")
TRAINING_INPUT_SHA256 = (
    sha256_file(TRAINING_ZIP) if TRAINING_ZIP else
    sha256_file(manifest_path) if TRAINING_INPUT and manifest_path and manifest_path.is_file() else None
)
EXPECTED_ARTIFACT_SHA256 = (
    {
        "fixed_adapter_checkpoint": sha256_file(ARTIFACTS["fixed_route_adapter.pt"]),
        "fixed_adapter_norms": sha256_file(ARTIFACTS["fixed_route_p99_norms.json"]),
        "bank_checkpoint": sha256_file(ARTIFACTS["session3_steering_bank.pt"]),
        "router_checkpoint": sha256_file(ARTIFACTS["session3_router.pt"]),
        "candidate_norms": sha256_file(ARTIFACTS["candidate_norms_p99.json"]),
        "static_bank_action": sha256_file(ARTIFACTS["session3_static_bank_action.json"]),
    }
    if INPUTS_READY else {}
)
ARTIFACT_FINGERPRINT = hashlib.sha256(
    json.dumps(EXPECTED_ARTIFACT_SHA256, sort_keys=True).encode("utf-8")
).hexdigest()[:12] if INPUTS_READY else "missing"
print({
    "repository": str(REPO_DIR),
    "evaluation_code_commit": EVALUATION_CODE_COMMIT,
    "training_code_commit": TRAINING_CODE_COMMIT,
    "training_input": str(TRAINING_INPUT) if TRAINING_INPUT else None,
    "input_mode": "zip" if TRAINING_ZIP else ("extracted_directory" if TRAINING_INPUT else "missing"),
    "artifacts_ready": INPUTS_READY,
    "artifact_fingerprint": ARTIFACT_FINGERPRINT,
    "artifacts": {key: str(value) for key, value in ARTIFACTS.items()},
    "gpus": subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True).stdout.splitlines(),
}, flush=True)


## Evaluation contract

This notebook is fixed to the official ARC-Challenge test split and samples 500 questions with seed 42. The same 500 test IDs and the same two demonstrations from the ARC-Challenge training split are used in both shot conditions; all four arms receive identical prompts within each condition.

AQuA-RAT is a strong math-specific multiple-choice alternative, but its official test split contains only 254 questions, so it cannot meet the requested 500-question comparison. ARC-Challenge is a widely used, challenging multiple-choice reasoning benchmark with enough official test items.


In [ ]:
DATASET = "arc"  # Official ARC-Challenge configuration in the benchmark runner.
DATASET_LABEL = "arc_challenge"
N_TEST = 500
SHOTS = [0, 2]
SEED = 42
MAX_NEW_TOKENS = 64
MAX_PROMPT_TOKENS = 1800

if DATASET != "arc" or DATASET_LABEL != "arc_challenge":
    raise ValueError("This notebook is intentionally fixed to the ARC-Challenge benchmark.")
if N_TEST != 500 or SHOTS != [0, 2]:
    raise ValueError("The configured study condition is exactly 500 questions at 0-shot and 2-shot.")
if not INPUTS_READY:
    print("Evaluation is blocked until a complete Stage B training bundle is attached.", flush=True)

RESULTS = WORK_ROOT / "ars_arc_challenge_results"
RESULTS.mkdir(parents=True, exist_ok=True)
previous_bundles = sorted({
    path
    for path in INPUT_ROOT.rglob(f"ars_{DATASET_LABEL}_n{N_TEST}_*shot_seed{SEED}_results.zip")
    if path.is_file()
})
if len(previous_bundles) > len(SHOTS):
    raise ValueError(
        "Attach at most one ARC-Challenge result ZIP per shot condition; "
        f"found: {[str(path) for path in previous_bundles]}"
    )
for previous_bundle in previous_bundles:
    merge_result_bundle(previous_bundle, RESULTS)

print({
    "dataset": "ARC-Challenge",
    "dataset_id": "allenai/ai2_arc",
    "configuration": "ARC-Challenge",
    "source_split": "test",
    "requested_questions_per_condition": N_TEST,
    "shots_in_order": SHOTS,
    "seed": SEED,
    "demonstration_split": "train",
    "arms": ["baseline", "steer_only", "static_bank", "router"],
    "training_bundle_ready": INPUTS_READY,
    "evaluation_code_commit": EVALUATION_CODE_COMMIT,
    "artifact_fingerprint": ARTIFACT_FINGERPRINT,
    "previous_result_bundles": [str(path) for path in previous_bundles],
}, flush=True)


## Results

After each shot condition, the notebook prints the four-arm accuracy, paired comparison, generation-performance, extraction-quality, and router-use summaries, then writes one independent ZIP:
- ars_arc_challenge_n500_0shot_seed42_results.zip
- ars_arc_challenge_n500_2shot_seed42_results.zip

Each ZIP contains the complete per-question predictions, aggregate record, run log/configuration, summaries, manifest, and evaluation-code snapshots. The final cell compares the completed 0-shot and 2-shot conditions. Re-running can reuse verified records and the MCQ runner can resume missing per-question predictions.


In [ ]:
from IPython.display import display
import pandas as pd
from scripts.research_notebook_analysis import build_benchmark_summary

CODE_FILES = [
    "scripts/run_session3_mcq_benchmark.py", "scripts/research_notebook_analysis.py",
    "scripts/research_notebook_utils.py", "src/evaluation/benchmark.py",
    "src/evaluation/answers.py", "src/evaluation/generation.py",
]

def condition_dir(shot):
    return RESULTS / ARTIFACT_FINGERPRINT / EVALUATION_CODE_COMMIT[:12] / DATASET / f"{shot}shot_n{N_TEST}"

def record_path(shot):
    return condition_dir(shot) / f"mcq_{DATASET}_{N_TEST}_seed{SEED}.json"

def is_complete(shot):
    path = record_path(shot)
    if not INPUTS_READY or not path.is_file():
        return False
    try:
        record = json.loads(path.read_text(encoding="utf-8"))
        return (
            record.get("dataset") == DATASET and record.get("seed") == SEED
            and record.get("requested_test_n") == N_TEST and record.get("test_n") == N_TEST
            and len(record.get("test_source_ids", [])) == N_TEST
            and set(record.get("shots", {})) == {str(shot)}
            and record.get("four_arm_complete") is True
            and record.get("provenance", {}).get("artifact_fingerprint") == ARTIFACT_FINGERPRINT
            and record.get("provenance", {}).get("evaluation_code_commit") == EVALUATION_CODE_COMMIT
        )
    except (OSError, ValueError, TypeError):
        return False

def package_condition(shot):
    out = condition_dir(shot)
    path = record_path(shot)
    if not is_complete(shot):
        raise RuntimeError(f"ARC-Challenge {shot}-shot did not complete all four arms on 500 paired questions.")
    summary = build_benchmark_summary(
        [out], out / "analysis", artifact_fingerprint=ARTIFACT_FINGERPRINT,
        evaluation_code_commit=EVALUATION_CODE_COMMIT,
    )
    metrics = pd.DataFrame(summary["arm_metrics"])
    if summary["record_count"] != 1 or set(metrics["arm"]) != {"baseline", "steer_only", "static_bank", "router"} or set(metrics["n"]) != {N_TEST}:
        raise RuntimeError(f"Invalid four-arm coverage for ARC-Challenge {shot}-shot: {metrics[['arm', 'n']].to_dict('records')}")
    print(f"ARC-Challenge {shot}-shot summary:")
    columns = [
        "shot", "arm", "n", "accuracy_pct", "paired_delta_pp_vs_baseline",
        "paired_ci95_pp_vs_baseline", "mcnemar_p_value_vs_baseline",
        "extraction_rate_pct", "mean_generation_latency_seconds", "questions_per_second",
        "output_tokens_per_second", "mean_new_tokens", "router_unique_routes",
        "router_most_used_route_share", "router_k_counts",
    ]
    display(metrics[[name for name in columns if name in metrics]].round(3))
    paired = pd.DataFrame(summary["paired_comparisons"])
    if not paired.empty:
        display(paired[[name for name in [
            "reference_arm", "arm", "n", "wins", "losses", "ties", "delta_pp", "ci95_pp", "mcnemar_p_value"
        ] if name in paired]].round(4))
    record = json.loads(path.read_text(encoding="utf-8"))
    atomic_write_json(out / "result_bundle_manifest.json", {
        "format": "ars_single_condition_result_bundle_v1", "dataset": "ARC-Challenge",
        "dataset_id": "allenai/ai2_arc", "configuration": "ARC-Challenge", "source_split": "test",
        "demonstration_split": "train", "requested_questions": N_TEST, "paired_questions": record["test_n"],
        "shot_count": shot, "seed": SEED, "arms": ["baseline", "steer_only", "static_bank", "router"],
        "four_arm_complete": True, "artifact_fingerprint": ARTIFACT_FINGERPRINT,
        "training_code_commit": TRAINING_CODE_COMMIT, "evaluation_code_commit": EVALUATION_CODE_COMMIT,
        "summary_files": ["analysis/benchmark_metrics_summary.json", "analysis/benchmark_metrics_summary.csv",
                          "analysis/benchmark_paired_comparisons.csv", "analysis/benchmark_route_usage.csv"],
    })
    files = {
        item: (out.relative_to(RESULTS) / item.relative_to(out)).as_posix()
        for item in out.rglob("*") if item.is_file()
    }
    for relative in CODE_FILES:
        source = REPO_DIR / relative
        if source.is_file():
            files[source] = f"code/{relative}"
    bundle = WORK_ROOT / f"ars_arc_challenge_n500_{shot}shot_seed{SEED}_results.zip"
    pack_files(files, bundle)
    print({"shot": shot, "complete": True, "questions": record["test_n"], "result_zip": str(bundle)}, flush=True)
    return bundle

def run_condition(shot):
    out = condition_dir(shot)
    if is_complete(shot):
        print(f"Reusing complete ARC-Challenge {shot}-shot condition.", flush=True)
        return package_condition(shot)
    if not INPUTS_READY:
        raise RuntimeError("Attach the complete Stage B training bundle before evaluation.")
    command = [
        sys.executable, "-u", str(REPO_DIR / "scripts" / "run_session3_mcq_benchmark.py"),
        "--dataset", DATASET, "--output_dir", str(out),
        "--candidate_layers", "8", "11", "12", "13", "15", "17", "18", "19", "20", "21", "22", "23", "24", "26",
        "--norm_calibration", str(ARTIFACTS["candidate_norms_p99.json"]),
        "--fixed_adapter_norm_calibration", str(ARTIFACTS["fixed_route_p99_norms.json"]),
        "--fixed_adapter_checkpoint", str(ARTIFACTS["fixed_route_adapter.pt"]),
        "--bank_checkpoint", str(ARTIFACTS["session3_steering_bank.pt"]),
        "--router_checkpoint", str(ARTIFACTS["session3_router.pt"]),
        "--static_action", str(ARTIFACTS["session3_static_bank_action.json"]),
        "--n_test", str(N_TEST), "--shots", str(shot), "--seed", str(SEED),
        "--max_new_tokens", str(MAX_NEW_TOKENS), "--max_prompt_tokens", str(MAX_PROMPT_TOKENS),
    ]
    code = run_live(command, cwd=REPO_DIR, log_path=out / "benchmark.log")
    path = record_path(shot)
    if code != 0 or not path.is_file():
        raise RuntimeError(f"ARC-Challenge {shot}-shot run failed (exit={code}); inspect {out / 'benchmark.log'}.")
    record = json.loads(path.read_text(encoding="utf-8"))
    record["provenance"] = {
        "training_code_commit": TRAINING_CODE_COMMIT, "evaluation_code_commit": EVALUATION_CODE_COMMIT,
        "training_input_sha256": TRAINING_INPUT_SHA256, "artifact_fingerprint": ARTIFACT_FINGERPRINT,
        "primary_shots": [0, 2],
    }
    atomic_write_json(path, record)
    return package_condition(shot)

condition_bundles = {}
for shot in SHOTS:
    condition_bundles[shot] = run_condition(shot)

final_summary = build_benchmark_summary(
    [RESULTS], RESULTS / "analysis", artifact_fingerprint=ARTIFACT_FINGERPRINT,
    evaluation_code_commit=EVALUATION_CODE_COMMIT,
)
final_metrics = pd.DataFrame(final_summary["arm_metrics"])
print("Final ARC-Challenge summary across completed conditions:")
if not final_metrics.empty:
    display(final_metrics[[name for name in [
        "shot", "arm", "n", "accuracy_pct", "paired_delta_pp_vs_baseline",
        "paired_ci95_pp_vs_baseline", "mcnemar_p_value_vs_baseline",
        "mean_generation_latency_seconds", "questions_per_second", "output_tokens_per_second",
        "router_unique_routes", "router_most_used_route_share", "router_k_counts",
    ] if name in final_metrics]].round(3))
print({
    "completed_shots": sorted({int(value) for value in final_metrics.get("shot", [])}) if not final_metrics.empty else [],
    "summary_json": str(RESULTS / "analysis" / "benchmark_metrics_summary.json"),
    "summary_csv": str(RESULTS / "analysis" / "benchmark_metrics_summary.csv"),
    "paired_comparisons_csv": str(RESULTS / "analysis" / "benchmark_paired_comparisons.csv"),
    "route_usage_csv": str(RESULTS / "analysis" / "benchmark_route_usage.csv"),
    "separate_result_zips": [str(path) for path in condition_bundles.values()],
    "measurement_definitions": final_summary["measure_definitions"],
}, flush=True)
